In [ ]:
import cv2
import numpy as np

import json

from pathlib import Path

In [ ]:
ROOT = Path.cwd().parent


GEOMETRY_FILE = (

ROOT /

"temp" /

"geometry_objects.json"

)


with open(

    GEOMETRY_FILE,

    encoding="utf-8"

) as f:


    geometry=json.load(f)



print(

"Objects:",

len(geometry)

)

In [ ]:
class CADObject:


    def __init__(

        self,

        obj_type,

        geometry,

        confidence

    ):


        self.type=obj_type

        self.geometry=geometry

        self.confidence=confidence



    def export(self):


        return {


        "type":
        self.type,


        "geometry":
        self.geometry,


        "confidence":
        self.confidence


        }

In [ ]:
def detect_wall(objects):


    walls=[]


    for obj in objects:


        if obj["type"]=="LINE":


            start=obj["start"]

            end=obj["end"]


            length=np.sqrt(

                (

                end[0]-start[0]

                )**2

                +

                (

                end[1]-start[1]

                )**2

            )


            if length > 200:


                walls.append(

                    CADObject(

                    "WALL",

                    obj,

                    0.75

                    )

                )


    return walls

In [ ]:
def detect_door(objects):


    doors=[]


    for obj in objects:


        if obj["type"]=="ARC":


            doors.append(

                CADObject(

                "DOOR",

                obj,

                0.70

                )

            )


    return doors

In [ ]:
def detect_window(objects):


    windows=[]


    for obj in objects:


        if obj["type"]=="LINE":


            start=obj["start"]

            end=obj["end"]


            dx=abs(

                end[0]-start[0]

            )


            dy=abs(

                end[1]-start[1]

            )


            if (

                dx > 50

                and

                dy < 10

            ):


                windows.append(

                    CADObject(

                    "WINDOW",

                    obj,

                    0.65

                    )

                )


    return windows

In [ ]:
def detect_room(objects):


    rooms=[]


    for obj in objects:


        if obj["type"]=="POLYLINE":


            rooms.append(

                CADObject(

                "ROOM",

                obj,

                0.80

                )

            )


    return rooms

In [ ]:
CAD_OBJECTS=[]


CAD_OBJECTS += detect_wall(

    geometry

)


CAD_OBJECTS += detect_door(

    geometry

)


CAD_OBJECTS += detect_window(

    geometry

)


CAD_OBJECTS += detect_room(

    geometry

)



print(

len(CAD_OBJECTS)

)

In [ ]:
for obj in CAD_OBJECTS:


    print(

        obj.type,

        obj.confidence

    )

In [ ]:
AI_FILE=(

ROOT /

"temp" /

"cad_objects.json"

)



with open(

    AI_FILE,

    "w",

    encoding="utf-8"

) as f:


    json.dump(

        [

        x.export()

        for x in CAD_OBJECTS

        ],

        f,

        indent=4

    )


print(AI_FILE)